# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [3]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent, AgentState
from lib.llm import LLM
import chromadb
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from lib.rag import RAG
from lib.state_machine import Run
from lib.vector_db import VectorStoreManager, CorpusLoaderService
from lib.evaluation import TestCase, AgentEvaluator, EvaluationResult
from pydantic import BaseModel, Field
import time
from tavily import TavilyClient
from typing import Dict
from datetime import datetime






In [4]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

In [5]:
# Environment variables are loaded in the setup cell above.

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
api_key = os.getenv("TAVILY_API_KEY")
client = TavilyClient(api_key=api_key)

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [6]:
# TODO: Create retrieve_game tool
# It should use the chroma client and collection you created in Part 1 (same path and name)


rag_llm = LLM(
    model="gpt-4o-mini",
    temperature=0.3,
)

chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")

games_rag = RAG(
    llm=rag_llm,
    vector_store = collection
)


@tool
def retrieve_game(question: str) -> dict:
    run = games_rag.invoke(question)
    state = run.get_final_state() or {}
    return {
        "answer": state.get("answer", ""),
        "documents": state.get("documents", []),
        "distances": state.get("distances", [])
    }


result:Run = games_rag.invoke("Who published Grand Theft Auto: San Andreas?")
print(result.get_final_state()["answer"])

result:Run = games_rag.invoke("When were Pokémon Gold and Silver released?")
print(result.get_final_state()["answer"])


result:Run = games_rag.invoke("What is Rockstar Games working on right now?")
print(result.get_final_state()["answer"])


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


[StateMachine] Starting: __entry__


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
Grand Theft Auto: San Andreas was published by Rockstar Games.
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
Pokémon Gold and Silver were released in 1999.
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
I don't know.


#### Evaluate Retrieval Tool

In [7]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

evaluator = AgentEvaluator()

class EvaluationReport(BaseModel):
    useful: bool = Field(
        description=(
            "True when the retrieved documents contain enough information "
            "to answer the user's question accurately."
        )
    )
    description: str = Field(
        description=(
            "Detailed explanation of why the documents are or are not "
            "sufficient to answer the question."
        )
    )
@tool
def evaluate_retrieval(question: str, retrieved_docs: list[str]) -> EvaluationReport:
    if not retrieved_docs:
        return EvaluationReport(useful=False, description="No documents were retrieved.")
    joined_docs = "\n\n".join(retrieved_docs)
    enough_context = len(joined_docs.strip()) > 80
    return EvaluationReport(
        useful=enough_context,
        description="Retrieved context is sufficient." if enough_context else "Retrieved context is too sparse; use web search."
    )



#### Game Web Search Tool

In [8]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Web search: searches the web with Tavily when the vector DB results aren't good enough
#    args:
#    - question: a question about game industry. 
#
#    Returns the search results, including each page's URL, so the answer can cite its source.

@tool
def web_search(
    query: str,
    search_depth: str = "advanced"
) -> Dict:
    """
    Search the web using Tavily API.

    Args:
        query: Search query.
        search_depth: Type of search - 'basic' or 'advanced'.

    Returns:
        A dictionary containing:
        - answer: Tavily-generated answer
        - results: Search results including title, URL, and content
        - search_metadata: Query and timestamp
    """

    api_key = os.getenv("TAVILY_API_KEY")
    client = TavilyClient(api_key=api_key)

    search_result = client.search(
        query=query,
        search_depth=search_depth,
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )

    # Explicitly include the page URL
    results = []

    for result in search_result.get("results", []):
        results.append({
            "title": result.get("title", ""),
            "url": result.get("url", ""),
            "content": result.get("content", ""),
            "score": result.get("score")
        })

    formatted_results = {
        "answer": search_result.get("answer", ""),
        "results": results,
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": query
        }
    }

    return formatted_results

### Agent

In [9]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

agent = Agent(
    model_name="gpt-4o-mini",
    tools=[
        retrieve_game,
        evaluate_retrieval,
        web_search
    ],
    instructions="""
You are an assistant specialized in answering questions about video games.

Follow this workflow:

1. ALWAYS search the local video game knowledge base first using
   retrieve_game.

2. After retrieving the documents, ALWAYS call evaluate_retrieval with:
   - the user's original question
   - the documents returned by retrieve_game

3. Inspect the EvaluationReport returned by evaluate_retrieval.

4. If useful=true:
   - Answer the user's question using ONLY the retrieved documents.
   - Do not perform a web search.

5. If useful=false:
   - Use web_search to find the missing information.
   - Base the final answer on the web search results.
   - Include the source page URL(s) returned by web_search.

6. Never invent information that is not supported by either the retrieved
   documents or the web search results.

7. Prefer the local game knowledge base whenever it contains enough
   information.

8. Give concise and direct answers.
"""
)

In [11]:
# TODO: Invoke your agent
# - When was Pokémon Gold and Silver released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X released for PlayStation 5?

questions = [
    "When were Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
    "What is Rockstar Games working on right now?"
]

for question in questions:
    print("=" * 70)
    print(f"Question: {question}")
    print("=" * 70)

    # Keep each question independent
    if hasattr(agent, "memory"):
        agent.memory.reset()

    try:
        # Run the agent
        run_object: Run = agent.invoke(question)

        # Get final state
        final_state: AgentState = run_object.get_final_state()

        if not final_state:
            print("ERROR: No final state returned.")
            continue

        messages = final_state.get("messages", [])

        if not messages:
            print("ERROR: No messages returned.")
            continue

        # -----------------------------
        # Show workflow / tools
        # -----------------------------
                
        print("\nWorkflow:")

        for msg in messages:
            tool_calls = getattr(msg, "tool_calls", None)

            if not tool_calls:
                continue

            for tool_call in tool_calls:
                tool_name = tool_call.function.name

                print(f"-> Calling tool: {tool_name}")
        # -----------------------------
        # Get final answer
        # -----------------------------
        final_response = ""

        for msg in reversed(messages):
            content = getattr(msg, "content", None)

            if not content:
                continue

            if isinstance(content, str):
                final_response = content
            else:
                final_response = str(content)

            break

        print("\nAnswer:")
        print(final_response)

    except Exception as error:
        print(
            f"\nERROR: {type(error).__name__}: {error}"
        )

    print()

Question: When were Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Workflow:
-> Calling tool: retrieve_game
-> Calling tool: evaluate_retrieval

Answer:
Pokémon Gold and Silver were released in 1999.

Question: Which one was the first 3D platformer Mario game?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[State

### (Optional) Advanced — Stand Out

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes
